[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/06_Advanced.ipynb)

# Module 6: Agent RL & Knowledge Distillation

In [ ]:
# @title 🔧 Environment Setup (Run this first!)
import os

gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print(f"GPU: {gpu_info[0]}")

if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')

!pip install -q transformers==4.57.6 datasets==3.6.0 jinja2==3.1.2 tokenizers

import sys
sys.path.insert(0, '.')
print("✅ Setup complete!")

## 📚 Overview

In this module we explore two advanced techniques:

**Part A — Agent RL for Tool Use**: Teaching the model to call external tools (calculators, APIs, weather services) through reinforcement learning with multi-turn rollouts.

**Part B — Knowledge Distillation**: Transferring knowledge from a larger “teacher” model to a smaller “student” model using soft probability distributions.

**Key source files:** `trainer/train_agent.py`, `dataset/lm_dataset.py` (AgentRLDataset), `trainer/train_distillation.py`

## Part A: Agent RL for Tool Use

### 1. Tool-Use in LLMs

Modern LLMs can call external tools to extend their capabilities:

```
User: "What's 15% of 340?"
  ↓
Model: <tool_call>{"name": "calculate_math", "arguments": {"expression": "340 * 0.15"}}</tool_call>
  ↓
System: <tool_response>{"result": "51.0"}</tool_response>
  ↓
Model: "15% of 340 is 51."
```

MiniMind supports 6 mock tools:
| Tool | Description |
|------|-------------|
| `calculate_math` | Evaluate math expressions |
| `unit_converter` | Convert between units |
| `get_current_weather` | Get weather for a city |
| `get_current_time` | Get time in a timezone |
| `get_exchange_rate` | Currency exchange rates |
| `translate_text` | Translate between languages |

In [ ]:
import json
import re
import math
import sys
sys.path.insert(0, '.')

# Tool definitions (JSON schema format)
TOOLS = [
    {"type": "function", "function": {
        "name": "calculate_math",
        "description": "Evaluate a math expression",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Math expression to evaluate"}
        }, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "get_current_weather",
        "description": "Get weather for a city",
        "parameters": {"type": "object", "properties": {
            "location": {"type": "string", "description": "City name"}
        }, "required": ["location"]}
    }},
    {"type": "function", "function": {
        "name": "get_exchange_rate",
        "description": "Get currency exchange rate",
        "parameters": {"type": "object", "properties": {
            "from_currency": {"type": "string"},
            "to_currency": {"type": "string"}
        }, "required": ["from_currency", "to_currency"]}
    }},
]

print("=== Tool Definitions ===")
for tool in TOOLS:
    fn = tool["function"]
    print(f"\n📦 {fn['name']}: {fn['description']}")
    print(f"   Parameters: {list(fn['parameters']['properties'].keys())}")

In [ ]:
# Mock tool execution (simulated responses)
WEATHER_DATA = {
    "北京": ("28°C", "晴"), "上海": ("15°C", "多云"),
    "Tokyo": ("12°C", "晴"), "New York": ("8°C", "多云"),
}
EXCHANGE_DATA = {("USD", "CNY"): 7.21, ("EUR", "CNY"): 7.85, ("GBP", "CNY"): 9.12}

MOCK_RESULTS = {
    "calculate_math": lambda args: {"result": str(eval(
        str(args.get("expression", "0")).replace("^", "**"),
        {"__builtins__": {}, "math": math}
    ))},
    "get_current_weather": lambda args: {
        "city": args.get("location"),
        "temperature": WEATHER_DATA.get(args.get("location"), ("22°C", "晴"))[0],
        "condition": WEATHER_DATA.get(args.get("location"), ("22°C", "晴"))[1]
    },
    "get_exchange_rate": lambda args: {
        "from": args.get("from_currency"), "to": args.get("to_currency"),
        "rate": EXCHANGE_DATA.get((args.get("from_currency"), args.get("to_currency")), 1.0)
    },
}

def parse_tool_calls(text):
    """Extract tool calls from model output."""
    calls = []
    for m in re.findall(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL):
        try:
            calls.append(json.loads(m.strip()))
        except:
            pass
    return calls

def execute_tool(name, args):
    """Execute a mock tool and return the result."""
    fn = MOCK_RESULTS.get(name)
    if not fn:
        return {"error": f"Unknown tool: {name}"}
    try:
        return fn(args)
    except Exception as e:
        return {"error": str(e)}

# Demo tool execution
print("=== Tool Execution Demo ===\n")

test_calls = [
    ("calculate_math", {"expression": "340 * 0.15"}),
    ("get_current_weather", {"location": "北京"}),
    ("get_exchange_rate", {"from_currency": "USD", "to_currency": "CNY"}),
]

for name, args in test_calls:
    result = execute_tool(name, args)
    print(f"🔧 {name}({args})")
    print(f"   → {result}\n")

### 2. Multi-Turn Rollout

The Agent RL training uses a **multi-turn rollout** loop:

```
Turn 1: Model generates response
  → If <tool_call> found: Parse and execute tool
  → Insert <tool_response> into conversation
  → Continue to Turn 2
Turn 2: Model generates with tool result
  → If another <tool_call>: Execute again
  → Otherwise: Final answer
```

The model is rewarded for:
- Correct tool selection (right tool for the question)
- Valid arguments (correct parameter names and values)
- Accurate final answer (matches ground truth)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('./model')

# Simulate a multi-turn agent conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant with access to tools."},
    {"role": "user", "content": "What is 15% of 340?"},
]

# Turn 1: Model would generate a tool call
model_output_turn1 = '<tool_call>\n{"name": "calculate_math", "arguments": {"expression": "340 * 0.15"}}\n</tool_call>'

# Parse and execute
tool_calls = parse_tool_calls(model_output_turn1)
print("=== Multi-Turn Agent Rollout ===\n")
print(f"User: What is 15% of 340?")
print(f"\nModel (Turn 1): {model_output_turn1}")

if tool_calls:
    call = tool_calls[0]
    result = execute_tool(call["name"], call["arguments"])
    tool_response = json.dumps(result, ensure_ascii=False)
    print(f"\nTool Response: <tool_response>{tool_response}</tool_response>")

    # Turn 2: Model generates final answer with tool result
    messages.append({"role": "assistant", "content": model_output_turn1})
    messages.append({"role": "system", "content": f"<tool_response>{tool_response}</tool_response>"})

    model_output_turn2 = "15% of 340 is 51.0"
    print(f"\nModel (Turn 2): {model_output_turn2}")
    messages.append({"role": "assistant", "content": model_output_turn2})

print(f"\n=== Full Conversation ({len(messages)} messages) ===")
for msg in messages:
    print(f"  [{msg['role']}] {msg['content'][:80]}")

### 3. Agent RL Dataset

The `AgentRLDataset` loads conversations with tool calls and ground truth answers:

```json
{
  "conversations": [
    {"role": "system", "content": "...", "tools": "[{...tool definitions...}]"},
    {"role": "user", "content": "What is the weather in Beijing?"},
    {"role": "assistant", "content": "<tool_call>{...}</tool_call>"},
    {"role": "system", "content": "<tool_response>{...}</tool_response>"},
    {"role": "assistant", "content": "The weather in Beijing is..."}
  ],
  "gt": {"tool": "get_current_weather", "expected": "28°C"}
}
```

The `gt` (ground truth) field is used to compute reward during RL training.

## Part B: Knowledge Distillation

### 4. Teacher-Student Framework

Knowledge distillation transfers knowledge from a **teacher** model to a **student** model:

```
Teacher (large, frozen)  →  Soft probabilities  →  Student (small, trainable)
                             T=4.0 softening
```

Why soft probabilities? They contain **“dark knowledge”** — information about which wrong answers are *almost right*:
- Hard label: "cat" = 1, everything else = 0
- Soft label: "cat" = 0.7, "kitten" = 0.2, "dog" = 0.05, ...

The student learns not just the right answer but the *relationships between answers*.

In [ ]:
import torch
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, temperature=1.0):
    """Compute KL divergence loss between student and teacher distributions.
    
    Temperature scaling softens the distributions:
    - T=1.0: Original (sharp) distributions
    - T=4.0: Softer distributions (more information about \"dark knowledge\")
    """
    with torch.no_grad():
        teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    kl = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
    return (temperature ** 2) * kl  # Scale by T² to match gradient magnitudes

# Demo: Effect of temperature
print("=== Temperature Scaling Demo ===\n")
logits = torch.tensor([[2.0, 1.0, 0.5, -1.0, -2.0]])

for T in [0.5, 1.0, 2.0, 4.0]:
    probs = F.softmax(logits / T, dim=-1)
    print(f"T={T:.1f}: {[f'{p:.3f}' for p in probs[0].tolist()]}")

print("\nHigher T → softer distribution → more 'dark knowledge' shared")

### 5. The Distillation Loss

The total loss blends hard labels (ground truth) and soft targets (teacher):

$$\mathcal{L} = \alpha \cdot \text{CE}(y, \hat{y}_{\text{student}}) + (1-\alpha) \cdot T^2 \cdot \text{KL}(\text{softmax}(\frac{z_s}{T}) \| \text{softmax}(\frac{z_t}{T}))$$

Where:
- $\alpha$ = weight for hard label loss (default: 0.5)
- $T$ = temperature (default: 1.5)
- $T^2$ = scaling factor to compensate for gradient magnitude changes
- $\text{CE}$ = cross-entropy with ground truth labels
- $\text{KL}$ = KL divergence from student to teacher

In [ ]:
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Teacher model (larger or MoE — here we simulate with same-size model)
teacher = MiniMindForCausalLM(config).to(device).eval()
teacher.requires_grad_(False)

# Student model (trainable)
student = MiniMindForCausalLM(config).to(device)

print(f"Teacher: {sum(p.numel() for p in teacher.parameters())/1e6:.1f}M params (frozen)")
print(f"Student: {sum(p.numel() for p in student.parameters())/1e6:.1f}M params (trainable)")

# Forward pass with distillation
input_ids = torch.randint(0, config.vocab_size, (2, 32)).to(device)
labels = input_ids.clone()

with torch.no_grad():
    teacher_logits = teacher(input_ids).logits[..., :-1, :].contiguous()

student_logits = student(input_ids).logits[..., :-1, :].contiguous()

# Compute losses
alpha = 0.5
temperature = 1.5

# Hard label CE loss
shift_labels = labels[..., 1:].contiguous()
ce_loss = F.cross_entropy(
    student_logits.view(-1, student_logits.size(-1)),
    shift_labels.view(-1),
    ignore_index=-100
)

# Soft target distillation loss
kd_loss = distillation_loss(
    student_logits.view(-1, student_logits.size(-1)),
    teacher_logits.view(-1, teacher_logits.size(-1)),
    temperature=temperature
)

# Blended loss
total_loss = alpha * ce_loss + (1 - alpha) * kd_loss

print(f"\n=== Distillation Loss Breakdown ===")
print(f"CE loss (hard labels):     {ce_loss.item():.4f}")
print(f"KD loss (soft targets):    {kd_loss.item():.4f}")
print(f"Total (α={alpha}):         {total_loss.item():.4f}")

### 6. When to Use Distillation

| Scenario | Recommendation |
|----------|---------------|
| Large teacher available | ✅ Distillation shines |
| No teacher model | Use SFT directly |
| Same architecture size | Can still benefit from MoE → Dense |
| Limited compute | LoRA + Distillation |
| Production deployment | Distill to smaller model for speed |

MiniMind's default setup: Distill from a MoE teacher model to a dense student model (same hidden size, fewer active parameters).

## ✏️ Exercises

1. **Tool call parsing**: Write a function that validates whether a tool call has all required parameters. Test it with correct and incorrect tool calls.

2. **New tool**: Add a `search_wikipedia` tool definition and mock execution function. How would you format the tool schema?

3. **Temperature experiment**: Plot the teacher's probability distribution at T=0.5, T=1.0, T=2.0, and T=4.0 for the same input. How does the "dark knowledge" change?

4. **Alpha sensitivity**: Try alpha=0.0 (pure distillation), alpha=0.5 (balanced), and alpha=1.0 (pure CE). Which gives the best student performance?

## 📝 Summary

In this module we covered:

- **Agent RL** — Teaching models to use tools via `<tool_call>`/`<tool_response>` format
- **6 mock tools** — Math, weather, currency, time, translation, unit conversion
- **Multi-turn rollout** — Generate → tool call → execute → respond → repeat
- **Tool reward signals** — Correct tool + valid arguments + accurate answer
- **Knowledge distillation** — Teacher → Student transfer via soft probabilities
- **Temperature scaling** — Higher T reveals more “dark knowledge”
- **Blended loss** — α·CE + (1-α)·T²·KL for combined hard/soft learning

🔜 **Next module:** Inference, deployment, and course wrap-up!